<a href="https://colab.research.google.com/github/eulaia/pibic-audiodescricao-llms/blob/main/Qwen2_5_VL_3B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate bitsandbytes qwen-vl-utils

import os
import torch

from datetime import datetime
from PIL import Image

from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig
)

from google.colab import drive
from IPython.display import display

# ==========================================================
# DRIVE
# ==========================================================

drive.mount('/content/drive')

# ==========================================================
# CONFIGURAÇÕES
# ==========================================================

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

FOLDER_PATH = "/content/drive/MyDrive/imagens"

PROMPT = """
Descreva a imagem de maneira objetiva e concisa, com foco em facilitar a interpretação
por um leitor de audiodescrição. Inclua os seguintes elementos:
Foco Principal: Identifique o principal sujeito ou objeto da imagem. Perspectiva da Foto:
Descreva a perspectiva da foto (ex: de frente, de lado, de cima, etc.) e o tipo de plano (ex: plano
geral, plano médio, close-up, etc.). Enquadramento: Explique como os elementos principais
estão posicionados e enquadrados na imagem. Plano de Fundo: Descreva o que está no fundo da
imagem e como ele contribui para a cena. Iluminação: Explique a iluminação da cena e como ela
afeta a percepção dos elementos na imagem. Contexto (se fornecido): Inclua detalhes sobre o
local e a época, se disponíveis. Preciso que a resposta seja no formato de parágrafo completo e
coerente, não faça em partes.
"""

# ==========================================================
# GPU
# ==========================================================

print("GPU disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ==========================================================
# QUANTIZAÇÃO
# ==========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# ==========================================================
# MODELO
# ==========================================================

print("\nCarregando modelo...")

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Modelo carregado!")

# ==========================================================
# GERAÇÃO
# ==========================================================

def gerar_descricao(imagem):

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": imagem
                },
                {
                    "type": "text",
                    "text": PROMPT
                }
            ]
        }
    ]

    texto = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[texto],
        images=[imagem],
        padding=True,
        return_tensors="pt"
    )

    device = next(model.parameters()).device

    inputs = {
        chave: valor.to(device)
        for chave, valor in inputs.items()
    }

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(
            inputs["input_ids"],
            generated_ids
        )
    ]

    resposta = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return resposta

# ==========================================================
# PROCESSAMENTO
# ==========================================================

def main():

    data_atual = datetime.now()

    imagens = [
        arquivo
        for arquivo in os.listdir(FOLDER_PATH)
        if arquivo.lower().endswith(
            (
                ".jpg",
                ".jpeg",
                ".png"
            )
        )
    ]

    print("\nRELATÓRIO DE GERAÇÃO DE AUDIODESCRIÇÃO")

    print(f"Data: {data_atual.strftime('%d/%m/%Y')}")
    print(f"Horário: {data_atual.strftime('%H:%M:%S')}")
    print(f"Modelo: {MODEL_ID}")
    print(f"Quantidade de imagens: {len(imagens)}")

    print("=" * 100)

    for arquivo in imagens:

        caminho_imagem = os.path.join(
            FOLDER_PATH,
            arquivo
        )

        print(f"\nProcessando: {arquivo}")

        imagem = Image.open(
            caminho_imagem
        )

        descricao = gerar_descricao(
            imagem
        )

        display(imagem)

        print("\nDescrição gerada:\n")
        print(descricao)

        print("\n" + "=" * 100)

    print("\nProcessamento finalizado.")

# ==========================================================
# EXECUÇÃO
# ==========================================================

main()